## Setup

In [ ]:
import numpy as np
from pathlib import Path
from itertools import product
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from scipy.stats import t
import itertools

source = Path("..\\results\\20260225_162434")
data_type = ["compressible", "incompressible"]
mode = ["compress", "decompress"]
languages = ["cpp", "go", "java", "python"]

results = []

for dt, m, lang in product(data_type, mode, languages):
    path = source / dt / m / lang / "metrics.csv"
    df = pd.read_csv(path)
    results.append(df)

df = pd.concat(results, ignore_index=True)
df["j/s"] = df["total_energy_j"] / df["wall_time_s"]
df

## Raw Visualizations

In [ ]:
# --- Violin plot ---
def raw_violin_plot(y, label):
    fig, axes = plt.subplots(2, 2, figsize=(10, 8))
    axes = axes.flatten()

    for ax, (dt, m) in zip(axes, product(data_type, mode)):
        subset = df[(df["dataset"] == dt) & (df["mode"] == m)]

        sns.violinplot(
            data=subset,
            x="lang",
            y=y,
            ax=ax
        )

        ax.set_title(f"{dt} - {m}")
        ax.set_xlabel("Language")
        ax.set_ylabel(label)

    plt.tight_layout()
    plt.show()

def select_data(ds, mode, lang):
    return df[(df["dataset"] == ds) & (df["mode"] == mode) & (df["lang"] == lang)]

In [ ]:
raw_violin_plot("total_energy_j", "Energy (j)")

In [ ]:
raw_violin_plot("energy_per_mb_j", "Energy per Input Size (j/mb)")

In [ ]:
raw_violin_plot("j/s", "Energy per Second (j/s)")

In [ ]:
raw_violin_plot("wall_time_s", "Time (millis)")

## Compression Ratio

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import t

def compression_ratio_table(df):
    # keep only compression runs (CR is only meaningful there)
    comp = df[df["mode"] == "compress"].copy()

    CR_COL = "compression_ratio"
    if CR_COL not in comp.columns:
        raise KeyError(
            f"Expected column '{CR_COL}' in df. Available columns: {list(comp.columns)}"
        )

    g = comp.groupby(["dataset", "lang"])[CR_COL]
    out = g.agg(n="count", mean_CR="mean", std_CR=lambda x: x.std(ddof=1)).reset_index()

    # 95% t-based CI (same style as energy)
    tcrit = out["n"].apply(lambda n: t.ppf(0.975, df=n-1) if n > 1 else np.nan)
    se = out["std_CR"] / np.sqrt(out["n"])
    ci_half = tcrit * se
    out["ci_lower_CR"] = out["mean_CR"] - ci_half
    out["ci_upper_CR"] = out["mean_CR"] + ci_half

    # compact formatting for report
    out["mean_CI_CR"] = out.apply(
        lambda r: f'{r["mean_CR"]:.5f} [{r["ci_lower_CR"]:.5f}, {r["ci_upper_CR"]:.5f}]',
        axis=1
    )

    out = out.sort_values(["dataset", "lang"]).reset_index(drop=True)
    return out

cr_df = compression_ratio_table(df)

print("\n=== Compression ratio summary (long form) ===")
print(cr_df[["dataset","lang","n","mean_CR","std_CR","ci_lower_CR","ci_upper_CR"]].to_string(index=False))

print("\n=== Compression ratio summary (pivot: mean [CI]) ===")
cr_pivot = cr_df.pivot_table(index="dataset", columns="lang", values="mean_CI_CR", aggfunc="first")
print(cr_pivot.to_string())

## Analysis

- RQ1: Do Python, Java, Go, and C++ differ significantly in energy consumption when performing identical gzip compression and decompression tasks under controlled conditions?
- RQ2: Are the observed energy differences consistent across data types (compressible vs. incompressible) and operations (compression vs. decompression), and how statistically significant are these differences?
- RQ3: Is the language that achieves the lowest runtime also the most energy efficient for gzip compression and decompression?

## Energy distribution per language (RQ1, RQ2)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 8))
axes = axes.flatten()

for ax, (dt, m) in zip(axes, product(data_type, mode)):
    subset = df[(df["dataset"] == dt) & (df["mode"] == m)]

    sns.violinplot(
        data=subset,
        x="lang",
        y="total_energy_j",
        ax=ax
    )

    ax.set_title(f"{dt} - {m}")
    ax.set_xlabel("Language")
    ax.set_ylabel("Energy (j)")

plt.tight_layout()
plt.title("Energy Distribution per Data Type, Task, and Language")
plt.show()

## Energy against Time (RQ3)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 8))
axes = axes.flatten()

handles_dict = {}

for ax, (dt, m) in zip(axes, product(data_type, mode)):
    subset = df[(df["mode"] == m) & (df["dataset"] == dt)]

    for lang, group in subset.groupby("lang"):
        sc = ax.scatter(
            group["wall_time_s"],
            group["total_energy_j"],
            label=lang
        )
        # store only first handle per language
        if lang not in handles_dict:
            handles_dict[lang] = sc

    ax.set_title(f"{dt} - {m}")
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("Energy (j)")

plt.tight_layout(rect=[0, 0, 1, 0.97])

fig.legend(
    handles_dict.values(),
    handles_dict.keys(),
    loc="upper right",
    ncol=len(handles_dict)
)

plt.suptitle("Energy vs Time")

plt.show()

## Mean Energy Usage (RQ1, RQ2)

In [ ]:
def show_mean_w_confidence(ax, m, ds, title):
    subset = df[(df["mode"] == m) & (df["dataset"] == ds)]
    grouped = subset.groupby("lang")["total_energy_j"]

    means = grouped.mean()
    std = grouped.std(ddof=1)
    n = grouped.count()

    # 95% CI using t-distribution
    ci = t.ppf(0.975, n-1) * (std / np.sqrt(n))

    ax.bar(means.index, means.values, yerr=ci.values, capsize=5)
    ax.set_ylabel("Mean Energy (J)")
    ax.set_xlabel("Language")
    ax.set_title(title)

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

show_mean_w_confidence(axes[0, 0], mode[0], data_type[0], "Compressible - Compress")
show_mean_w_confidence(axes[0, 1], mode[1], data_type[0], "Compressible - Decompress")

show_mean_w_confidence(axes[1, 0], mode[0], data_type[1], "Incompressible - Compress")
show_mean_w_confidence(axes[1, 1], mode[1], data_type[1], "Incompressible - Decompress")

fig.suptitle("Mean Energy Usage")
plt.tight_layout()
plt.show()

## Effect sizes (RQ2)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from itertools import product

def cohens_d(x, y):
    sx, sy = np.std(x, ddof=1), np.std(y, ddof=1)
    s_pooled = np.sqrt((sx**2 + sy**2) / 2)
    return (np.mean(y) - np.mean(x)) / s_pooled

languages = sorted(df["lang"].unique())
datasets = sorted(df["dataset"].unique())
modes = sorted(df["mode"].unique())

fig, axes = plt.subplots(len(datasets), len(modes),
                         figsize=(5 * len(modes), 4 * len(datasets)))

# Ensure 2D indexing
if len(datasets) == 1:
    axes = np.array([axes])
if len(modes) == 1:
    axes = axes.reshape(-1, 1)

for i, ds in enumerate(datasets):
    for j, m in enumerate(modes):
        ax = axes[i, j]
        subset = df[(df["dataset"] == ds) & (df["mode"] == m)]

        mat = np.zeros((len(languages), len(languages)))

        for a, lang_a in enumerate(languages):
            for b, lang_b in enumerate(languages):
                if a == b:
                    mat[a, b] = 0
                else:
                    x = subset[subset["lang"] == lang_a]["total_energy_j"]
                    y = subset[subset["lang"] == lang_b]["total_energy_j"]

                    if len(x) > 1 and len(y) > 1:
                        mat[a, b] = cohens_d(x, y)
                    else:
                        mat[a, b] = np.nan

        im = ax.imshow(mat, vmin=-2, vmax=2)
        ax.set_xticks(range(len(languages)))
        ax.set_yticks(range(len(languages)))
        ax.set_xticklabels(languages)
        ax.set_yticklabels(languages)
        ax.set_title(f"{ds} - {m}")

fig.colorbar(im, ax=axes.ravel().tolist(), label="Cohen's d (y - x)")
plt.show()

## Energy per MB (RQ1, RQ2)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 8))
axes = axes.flatten()

for ax, (dt, m) in zip(axes, product(data_type, mode)):
    subset = df[(df["dataset"] == dt) & (df["mode"] == m)]

    sns.violinplot(
        data=subset,
        x="lang",
        y="energy_per_mb_j",
        ax=ax
    )

    ax.set_title(f"{dt} - {m}")
    ax.set_xlabel("Language")
    ax.set_ylabel("Energy per MB (j/MB)")

plt.tight_layout()
fig.suptitle("Energy per MB")
plt.show()

## Statistical Analysis (RQ1, RQ2)

In [ ]:
results = []
for ds, m, lang in product(data_type, mode, languages):
    subset = df[
        (df["dataset"] == ds) &
        (df["mode"] == m) &
        (df["lang"] == lang)
    ]["total_energy_j"]

    n = len(subset)
    mean = subset.mean()
    std = subset.std(ddof=1)

    t_crit = t.ppf(0.975, df=n-1)
    ci_half = t_crit * std / np.sqrt(n)

    results.append({
        "dataset": ds,
        "mode": m,
        "lang": lang,
        "mean_J": mean,
        "std_J": std,
        "ci_lower_J": mean - ci_half,
        "ci_upper_J": mean + ci_half
    })

results_df = pd.DataFrame(results)
results_df = results_df.sort_values(["dataset", "mode", "lang"])

results_df

In [ ]:
from scipy.stats import t

def energy_summary_table(df):
    # group by condition + language
    g = df.groupby(["dataset", "mode", "lang"])["total_energy_j"]

    out = g.agg(n="count", mean_J="mean", std_J=lambda x: x.std(ddof=1)).reset_index()

    # 95% t-based CI
    tcrit = out["n"].apply(lambda n: t.ppf(0.975, df=n-1) if n > 1 else np.nan)
    se = out["std_J"] / np.sqrt(out["n"])
    ci_half = tcrit * se

    out["ci_lower_J"] = out["mean_J"] - ci_half
    out["ci_upper_J"] = out["mean_J"] + ci_half

    # nice formatting column for the report
    out["mean_CI_J"] = out.apply(
        lambda r: f'{r["mean_J"]:.2f} [{r["ci_lower_J"]:.2f}, {r["ci_upper_J"]:.2f}]',
        axis=1
    )

    # sort for readability
    out = out.sort_values(["dataset", "mode", "lang"]).reset_index(drop=True)
    return out

summary_df = energy_summary_table(df)

print("\n=== Energy summary (long form) ===")
print(summary_df[["dataset","mode","lang","n","mean_J","std_J","ci_lower_J","ci_upper_J"]].to_string(index=False))

print("\n=== Energy summary (pivot: mean [CI]) ===")
pivot = summary_df.pivot_table(
    index=["dataset","mode"],
    columns="lang",
    values="mean_CI_J",
    aggfunc="first"
)
print(pivot.to_string())

summary_df.to_csv(source / "energy_summary_table.csv", index=False)
pivot.to_csv(source / "energy_summary_pivot.csv")

## Percent Change (RQ1)

In [ ]:
results = []

for (ds, mode), group in df.groupby(["dataset", "mode"]):
    means = group.groupby("lang")["total_energy_j"].mean()

    for a, b in itertools.combinations(means.index, 2):
        mean_a = means[a]
        mean_b = means[b]

        pct_change = ((mean_b - mean_a) / mean_a) * 100

        results.append({
            "dataset": ds,
            "mode": mode,
            "baseline_lang": a,
            "compare_lang": b,
            "percent_change": pct_change
        })

pd.DataFrame(results)

## All Effect Size Analysis

In [ ]:
import numpy as np
import pandas as pd
import itertools

def cohens_d(x, y):
    # Same definition you used in the heatmap cell
    sx, sy = np.std(x, ddof=1), np.std(y, ddof=1)
    s_pooled = np.sqrt((sx**2 + sy**2) / 2)
    return (np.mean(y) - np.mean(x)) / s_pooled

rows = []
langs = sorted(df["lang"].unique())

for (ds, m), sub in df.groupby(["dataset", "mode"]):
    for lang_a, lang_b in itertools.combinations(langs, 2):
        x = sub[sub["lang"] == lang_a]["total_energy_j"].dropna().to_numpy()
        y = sub[sub["lang"] == lang_b]["total_energy_j"].dropna().to_numpy()

        if len(x) < 2 or len(y) < 2:
            continue

        mean_a = x.mean()
        mean_b = y.mean()

        mean_diff_J = mean_b - mean_a
        pct_change = (mean_diff_J / mean_a) * 100.0
        d = cohens_d(x, y)

        rows.append({
            "dataset": ds,
            "mode": m,
            "lang_a": lang_a,
            "lang_b": lang_b,
            "cohens_d_(b-a)": d,
            "mean_diff_J_(b-a)": mean_diff_J,
            "pct_change_%_(b_vs_a)": pct_change,
            "mean_a_J": mean_a,
            "mean_b_J": mean_b,
        })

effect_table = pd.DataFrame(rows).sort_values(
    ["dataset", "mode", "lang_a", "lang_b"]
).reset_index(drop=True)

print(effect_table.to_string(index=False))

## Pearson correlation between runtime and energy (RQ3)

In [ ]:
from scipy.stats import pearsonr
import numpy as np
import pandas as pd

rows = []
for (ds, mode, lang), g in df.groupby(["dataset", "mode", "lang"]):
    x = g["wall_time_s"].to_numpy()
    y = g["total_energy_j"].to_numpy()
    if len(g) >= 3 and np.std(x) > 0 and np.std(y) > 0:
        r, p = pearsonr(x, y)
    else:
        r, p = np.nan, np.nan
    rows.append({"dataset": ds, "mode": mode, "lang": lang, "pearson_r": r, "p_value": p})

within_lang_corr = pd.DataFrame(rows).sort_values(["dataset","mode","lang"])
within_lang_corr

## Shapiro-Wilk (RQ2)

In [ ]:
from scipy.stats import shapiro

results = []

for (ds, m), group in df.groupby(["dataset", "mode"]):
    for lang, g in group.groupby("lang"):
        values = g["total_energy_j"].dropna()

        if len(values) >= 3:  # Shapiro requires at least 3
            W, p = shapiro(values)

            results.append({
                "dataset": ds,
                "mode": m,
                "lang": lang,
                "W": W,
                "p_value": p,
                "normal_0.05": p > 0.05
            })

pd.DataFrame(results)

## Welch's t-test between languages per mode/data type (RQ2)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import ttest_ind
from statsmodels.stats.multitest import multipletests

languages = sorted(df["lang"].unique())
datasets = sorted(df["dataset"].unique())
modes = sorted(df["mode"].unique())

pairwise_rows = []

fig, axes = plt.subplots(len(datasets), len(modes),
                         figsize=(5 * len(modes), 4 * len(datasets)))

if len(datasets) == 1:
    axes = np.array([axes])
if len(modes) == 1:
    axes = axes.reshape(-1, 1)

for i, ds in enumerate(datasets):
    for j, m in enumerate(modes):
        ax = axes[i, j]
        subset = df[(df["dataset"] == ds) & (df["mode"] == m)]

        p_mat = np.full((len(languages), len(languages)), np.nan)

        pvals = []
        pairs = []
        raw_map = {}

        # --- collect raw p-values ---
        for a in range(len(languages)):
            for b in range(a + 1, len(languages)):
                lang_a = languages[a]
                lang_b = languages[b]

                x = subset[subset["lang"] == lang_a]["total_energy_j"].dropna()
                y = subset[subset["lang"] == lang_b]["total_energy_j"].dropna()

                if len(x) > 1 and len(y) > 1:
                    _, p_raw = ttest_ind(x, y, equal_var=False)
                    pvals.append(p_raw)
                    pairs.append((a, b))
                    raw_map[(a, b)] = p_raw

        # --- Holm correction + fill matrix + store exact p-values ---
        if pvals:
            reject, pvals_corr, _, _ = multipletests(
                pvals,
                alpha=0.05,
                method="holm"
            )

            for (a, b), p_corr, rej in zip(pairs, pvals_corr, reject):
                # fill symmetric matrix for heatmap
                p_mat[a, b] = p_corr
                p_mat[b, a] = p_corr

                pairwise_rows.append({
                    "dataset": ds,
                    "mode": m,
                    "lang_a": languages[a],
                    "lang_b": languages[b],
                    "p_holm": p_corr,
                    "reject_0.05": bool(rej),
                })

        im = ax.imshow(p_mat, vmin=0, vmax=0.05)
        ax.set_xticks(range(len(languages)))
        ax.set_yticks(range(len(languages)))
        ax.set_xticklabels(languages)
        ax.set_yticklabels(languages)
        ax.set_title(f"{ds} - {m}")

fig.colorbar(im, ax=axes.ravel().tolist(), label="Welch p-value (Holm-corrected)")
fig.suptitle("Welch's t-test between languages per mode/data type (Holm-corrected)")
plt.show()

welch_table = pd.DataFrame(pairwise_rows).sort_values(["dataset","mode","p_holm","lang_a","lang_b"])
print(welch_table.to_string(index=False))